In [1]:
import os
import gzip
from collections import Counter
from typing import List
import pandas as pd

In [2]:
in_dir1 = "../../data/Sitara_scACCESS_mouseAirwayCell_112125/barcodes"
in_dir2 = "../../data/Sitara_scACCESS_mouseAirwayCell_121725/barcodes"

out_dir = "../../results/04_single_cell_access_atac/04_barcode_stat"

os.makedirs(out_dir, exist_ok=True)

In [3]:
def open_maybe_gz(path: str):
    """Open a plain or gzipped text file in text mode."""
    if path.endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")

def pass1_count_exact(fastq_paths: List[str], corrected_dict: dict) -> Counter:
    """
    First pass across ALL fastqs:
      count exact matches of observed barcodes to whitelist.
    """
    counts = Counter()
    for path in fastq_paths:
        print(f"Processing {path}")
        with open_maybe_gz(path) as f:
            while True:
                header = f.readline()
                if not header:
                    break
                seq = f.readline().strip()
                plus = f.readline()
                qual = f.readline().strip()
                if not qual:
                    break

                counts[seq] += 1
    return counts

In [4]:
barcode_list = ['Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in1.fastq.gz',
                'Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in2.fastq.gz', 
                'Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in3.fastq.gz', 
                'Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in4.fastq.gz']

In [5]:
# read corrected barcodes
df = pd.read_csv(f"../../results/04_single_cell_access_atac/03_correct_barcode/barcode.tsv", 
                 sep="\t", header=None, names=["raw", "corrected"])
# convert to dict
corrected_dict = dict(zip(df["raw"], df["corrected"]))

In [6]:
for path in [in_dir1, in_dir2]:
    fastq_paths = [f"{path}/{bc}" for bc in barcode_list]
    counts = pass1_count_exact(fastq_paths, corrected_dict)

Processing ../../data/Sitara_scACCESS_mouseAirwayCell_112125/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in1.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_112125/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in2.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_112125/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in3.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_112125/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in4.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_121725/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in1.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_121725/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in2.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_121725/barcodes/Sitara_1SR_SingleCellmouse_ATAC-ACCESS_10162025_in3.fastq.gz
Processing ../../data/Sitara_scACCESS_mouseAirwayCell_121725/barcodes/Sitara_1SR_Si

In [7]:
df_barcode = pd.DataFrame.from_dict(counts, orient='index', columns=['count'])

In [8]:
df_whiltelist = pd.read_csv("../../data/10xGenomics/737K-cratac-v1.txt", header=None, names=["barcode"])

In [9]:
df_valid_barcodes = df_barcode.loc[df_barcode.index.isin(df_whiltelist['barcode'])]
df_non_valid_barcodes = df_barcode.loc[~df_barcode.index.isin(df_whiltelist['barcode'])]

In [10]:
df_non_valid_barcodes["count"].sum()

np.int64(37963537)

In [11]:
df_valid_barcodes["count"].sum()

np.int64(359979787)

In [12]:
frac_valid_barcode = df_valid_barcodes["count"].sum() / df_barcode["count"].sum()
print(f"Fraction of valid barcodes: {frac_valid_barcode:.4f}")

Fraction of valid barcodes: 0.9046


In [13]:
df_valid_barcodes.loc[:, 'is_valid_barcode'] = 1
df_non_valid_barcodes.loc[:, 'is_valid_barcode'] = 0

/data/pinello/tmp/zl_tmp/ipykernel_3241785/759626667.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valid_barcodes.loc[:, 'is_valid_barcode'] = 1
/data/pinello/tmp/zl_tmp/ipykernel_3241785/759626667.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_non_valid_barcodes.loc[:, 'is_valid_barcode'] = 0


In [14]:
df_barcode = pd.concat([df_valid_barcodes, df_non_valid_barcodes], axis=0)
df_barcode["barcode"] = df_barcode.index

In [15]:
df_barcode

,count,is_valid_barcode,barcode
GCTTTCGAGTTTACGC,849524,1,GCTTTCGAGTTTACGC
GCCAGACTCTTGGCCT,228547,1,GCCAGACTCTTGGCCT
TTGGTCCGTCAGACGA,333360,1,TTGGTCCGTCAGACGA
GCGATTACATTTGTTC,466966,1,GCGATTACATTTGTTC
GCCATAACACAGTAGG,823113,1,GCCATAACACAGTAGG
...,...,...,...
AAAAGAACAAGGAAAC,1,0,AAAAGAACAAGGAAAC
CCTATTCCTCAGCCCA,1,0,CCTATTCCTCAGCCCA
GGTTTATAGGCAGTTG,1,0,GGTTTATAGGCAGTTG
TGCCTCACANTGCACT,1,0,TGCCTCACANTGCACT


In [16]:
df_barcode.to_csv(f"{out_dir}/barcode_stat.tsv", 
                  sep="\t", header=True, index=False)